In [1]:
import BECancerResistome
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import math
from sklearn.preprocessing import OrdinalEncoder

# Load Data

## Samples

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
annotated_control_df = pd.read_csv("data/8_Celular_Context/zscores-unambiguous-VEPannotated-processed-control-with-gene-expression.csv")

/var/folders/ry/v0cp3ptj55qfs_pd6_cghmzm0000gn/T/ipykernel_74576/850768828.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  annotated_control_df = pd.read_csv("data/8_Celular_Context/zscores-unambiguous-VEPannotated-processed-control-with-gene-expression.csv")


In [4]:
vep_annotated_control_df = pd.read_csv("data/6_VEP/zscores-unambiguous-VEPannotated-control.csv")

/var/folders/ry/v0cp3ptj55qfs_pd6_cghmzm0000gn/T/ipykernel_24323/2333214704.py:1: DtypeWarning: Columns (7,26,111,112,113,114,115) have mixed types. Specify dtype option on import or set low_memory=False.
  vep_annotated_control_df = pd.read_csv("data/6_VEP/zscores-unambiguous-VEPannotated-control.csv")


In [5]:
foldx_sample = pd.read_csv('data/9_ProtVar/Samples/2025.02.10_foldx_energy_sample.csv')
pockets_sample = pd.read_csv('data/9_ProtVar/Samples/2024.05.28_pockets_sample.tsv', sep='\t')
interfaces_sample = pd.read_csv('data/9_ProtVar/Samples/2024.05.28_interface_summary_sample.tsv', sep='\t')

In [187]:
foldx_sample.head()

,uniprot_accession,uniprot_position,alphafold_fragment_id,alphafold_fragment_position,wild_type,mutated_type,foldx_ddg,plddt
0,A0A075B6S0,1,F1,1,N,A,-0.001835,56.9
1,A0A075B6S0,1,F1,1,N,C,-0.011188,56.9
2,A0A075B6S0,1,F1,1,N,D,-0.041779,56.9
3,A0A075B6S0,1,F1,1,N,E,-0.024084,56.9
4,A0A075B6S0,1,F1,1,N,F,0.324027,56.9


In [27]:
pockets_sample.head()

,struct_id,pocket_id,pocket_rad_gyration,pocket_energy_per_vol,pocket_buriedness,pocket_resid,pocket_plddt_mean,pocket_score_combined_scaled
0,A0A024R1R8-F1,1,4.042788,0.316535,0.772959,"{21,22,23,24,25,26,28,29,32}",83.937778,283.034096
1,A0A024R1R8-F1,2,3.175737,0.347111,0.808219,"{12,13,14,15,16,17}",61.206667,102.718057
2,A0A024RBG1-F1,1,7.310256,0.435597,0.856184,"{2,3,4,5,6,7,8,9,10,18,20,21,22,39,40,41,42,47...",89.456190,979.457587
3,A0A024RBG1-F1,2,6.350910,0.389675,0.814896,"{54,57,58,60,61,62,64,65,67,68,69,73,74,75,76,...",83.186923,938.222063
4,A0A024RBG1-F1,3,3.827945,0.378204,0.806045,"{1,2,3,4,5,6,109,110,112,113,114}",77.053636,422.703190


In [28]:
interfaces_sample.head()

,interaction_id,pdockq,uniprot_id1,uniprot_id2,chain1,chain2,ifresid1,ifresid2,sources,n_references,pdb
0,O75106_Q16853,0.74,O75106,Q16853,A,B,"R169,A203,A204,V205,H206,L212,R213,W220,N226,I...","P39,V209,L218,Q219,W226,N232,I233,S234,G235,A2...","BioGRID,humap,intact,string",2,O75106/O75106_Q16853.pdb
1,Q15118_Q15118,0.73,Q15118,Q15118,A,B,"S53,P54,P56,Y179,D182,R183,M186,L255,A257,H304...","S53,P54,P56,Y179,D182,R183,M186,E253,L255,A257...","BioGRID,intact",2,Q15118/Q15118_Q15118.pdb
2,P11142_Q92598,0.73,P11142,Q92598,A,B,"K25,E27,I28,A30,N31,D32,Q33,G34,R36,E48,L50,D5...","R19,A27,N28,E29,F30,S31,R33,N54,T58,Y184,R261,...","BioGRID,corum,humap,intact,otar,string,xlinkdb",9,P11142/P11142_Q92598.pdb
3,Q13326_Q16585,0.73,Q13326,Q16585,A,B,"V40,L41,L43,L44,L47,V48,N50,L51,T54,I55,L58,F6...","V68,I69,L71,L72,L75,A76,I78,N79,I82,I86,M100,F...","corum,otar,string",0,Q13326/Q13326_Q16585.pdb
4,Q13326_Q92629,0.73,Q13326,Q92629,A,B,"K33,L36,Y37,V40,L41,L43,L44,L47,V48,N50,L51,T5...","R30,K31,C33,L34,F37,V38,L40,L41,L44,I45,V47,N4...","corum,string",0,Q13326/Q13326_Q92629.pdb


In [4]:
annotated_control_df.head()

,Guide,Editor,Gene,Cell_Line,Drug,zscore,Hit_class,Source,Target Transcript ID,RefSeq match transcript (MANE Select),Amino Acid Edits,Mutation_Category_enc,IMPACT_enc,TSL,SIFT_pathogenicity,PolyPhen_pathogenicity,REVEL,ClinPred,EVE_SCORE,AlphaMissense_score,BayesDel_noAF_score,DANN_score,DEOGEN2_score,ESM1b_score,Eigen-PC-phred_coding,Eigen-PC-raw_coding,GERP++_NR,GERP++_RS,LIST-S2_score,MPC_rankscore,MPC_score,MVP_score,MetaRNN_score,MetaSVM_score,MutFormer_score,MutationAssessor_score,PROVEAN_score,PrimateAI_pred_enc,VARITY_R_LOO_score,bStatistic,fathmm-XF_coding_score,gMVP_score,phastCons100way_vertebrate,phyloP100way_vertebrate,CADD_PHRED,MaxEntScan_alt,MaxEntScan_diff,MaxEntScan_ref,SpliceAI_pred_DP_AG,SpliceAI_pred_DP_AL,SpliceAI_pred_DP_DG,SpliceAI_pred_DP_DL,SpliceAI_pred_DS_AG,SpliceAI_pred_DS_AL,SpliceAI_pred_DS_DG,SpliceAI_pred_DS_DL,BLOSUM62,LOEUF,mutfunc_exp,ada_score,Gene_expression_voom
0,AAAAAACATCGTAGTAGCAG,CBE,RICTOR,A375,PIC,0.274277,non-hit,NaN,ENST00000357387.8,NM_152756.5,His1109Tyr,3.0,2.0,1.0,1.0,0.000,0.214,0.463040,NaN,0.0685,-0.402171,0.970933,0.045001,-4.388,5.314964,0.516773,5.75,5.75,0.917608,0.44121,0.445060,0.093490,0.179100,-1.0259,0.000785,0.345,-0.22,1.0,0.094767,226.0,0.800526,NaN,1.000,5.938,20.400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.305,NaN,NaN,5.531900
1,AAAAAACTCAACAGCAGTGA,CBE,NRAS,A375,PIC,0.399691,non-hit,NaN,ENST00000369535.5,NM_002524.5,Leu171Phe,3.0,2.0,1.0,0.1,NaN,0.141,0.413488,NaN,0.1149,-0.222424,0.993713,0.120039,-5.943,3.683551,0.294249,5.86,5.86,0.865513,0.74516,1.008632,0.619943,0.190982,-0.9613,0.907693,1.385,-1.28,1.0,0.072730,819.0,0.580949,0.628916,1.000,4.055,23.400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.550,NaN,NaN,6.702081
2,AAAAAACTGGCCAGGTGAGC,CBE,PIK3CB,A375,PIC,0.399178,non-hit,NaN,ENST00000674063.1,NM_006219.3,Thr930Ile,3.0,2.0,NaN,0.0,0.862,NaN,0.373381,NaN,NaN,-0.058094,0.998396,0.879846,-10.191,4.891242,0.469329,5.63,3.78,NaN,0.73854,NaN,NaN,NaN,0.6434,0.529280,3.705,NaN,0.0,0.365695,856.0,0.310513,NaN,0.998,2.094,25.400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,0.319,NaN,NaN,5.786491
3,AAAAAACTGTTTGGGACCTC,CBE,EGFR,A375,PIC,-0.801011,non-hit,NaN,ENST00000275493.7,NM_005228.5,Leu480Leu,2.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.051,NaN,NaN,NaN,21.0,-44.0,-45.0,45.0,0.0,0.0,0.0,0.0,NaN,0.475,-0.01879,NaN,3.875409
4,AAAAAATCCAGCGTCTAAGC,CBE,MYC,A375,PIC,-0.379377,non-hit,NaN,ENST00000621592.8,NM_002467.6,Asp2Asn,3.0,2.0,1.0,0.0,0.199,NaN,0.453567,NaN,0.1399,-0.438067,0.984375,NaN,NaN,1.792897,-0.197017,4.31,2.43,0.397160,NaN,NaN,0.733174,0.215208,-0.7405,0.036037,NaN,NaN,1.0,NaN,951.0,0.226082,0.128201,1.000,1.507,22.100,NaN,NaN,NaN,16.0,-14.0,10.0,26.0,0.0,0.0,0.0,0.0,1.0,0.128,NaN,NaN,3.687539


In [19]:
#Create a txt file with all target transcript IDs
transcript_ids = annotated_control_df['Target Transcript ID'].unique()
with open('data/9_ProtVar/Ensembl_transcript_ids.txt', 'w') as f:
    for transcript_id in transcript_ids:
        f.write(f"{transcript_id}\n")

In [14]:
#Open excel file with mapping of transcript IDs to UniProt IDs
transcript_to_uniprot_df = pd.read_excel('data/9_ProtVar/idmapping_2025_10_27.xlsx')

/Users/carolinapinto/Desktop/Tese/BECancerResistome/venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [15]:
transcript_to_uniprot_df

,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length
0,ENST00000357387.8,Q6R327,reviewed,RICTR_HUMAN,Rapamycin-insensitive companion of mTOR (AVO3 ...,RICTOR KIAA1999,Homo sapiens (Human),1708
1,ENST00000369535.5,P01111,reviewed,RASN_HUMAN,GTPase NRas (EC 3.6.5.2) (Transforming protein...,NRAS HRAS1,Homo sapiens (Human),189
2,ENST00000674063.1,P42338,reviewed,PK3CB_HUMAN,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",PIK3CB PIK3C1,Homo sapiens (Human),1070
3,ENST00000275493.7,P00533,reviewed,EGFR_HUMAN,Epidermal growth factor receptor (EC 2.7.10.1)...,EGFR ERBB ERBB1 HER1,Homo sapiens (Human),1210
4,ENST00000621592.8,P01106,reviewed,MYC_HUMAN,Myc proto-oncogene protein (Class E basic heli...,MYC BHLHE39,Homo sapiens (Human),454
5,ENST00000251849.9,P04049,reviewed,RAF1_HUMAN,RAF proto-oncogene serine/threonine-protein ki...,RAF1 RAF,Homo sapiens (Human),648
6,ENST00000358273.9,P21359,reviewed,NF1_HUMAN,Neurofibromin (Neurofibromatosis-related prote...,NF1,Homo sapiens (Human),2839
7,ENST00000361445.9,P42345,reviewed,MTOR_HUMAN,Serine/threonine-protein kinase mTOR (EC 2.7.1...,MTOR FRAP FRAP1 FRAP2 RAFT1 RAPT1,Homo sapiens (Human),2549
8,ENST00000320031.13,P26006,reviewed,ITA3_HUMAN,Integrin alpha-3 (CD49 antigen-like family mem...,ITGA3 MSK18,Homo sapiens (Human),1051
9,ENST00000448116.7,P29353,reviewed,SHC1_HUMAN,SHC-transforming protein 1 (SHC-transforming p...,SHC1 SHC SHCA,Homo sapiens (Human),583


### Check if protein sequence from the Ensembl Transcript and the UniProt protein match

In [147]:
ensembl_sequence = """MAARRRRSTGGGRARALNESKRVNNGNTAPEDSSPAKKTRRCQRQESKKMPVAGGKANKD
RTEDKQDESVKALLLKGKAPVDPECTAKVGKAHVYCEGNDVYDVMLNQTNLQFNNNKYYL
IQLLEDDAQRNFSVWMRWGRVGKMGQHSLVACSGNLNKAKEIFQKKFLDKTKNNWEDREK
FEKVPGKYDMLQMDYATNTQDEEETKKEESLKSPLKPESQLDLRVQELIKLICNVQAMEE
MMMEMKYNTKKAPLGKLTVAQIKAGYQSLKKIEDCIRAGQHGRALMEACNEFYTRIPHDF
GLRTPPLIRTQKELSEKIQLLEALGDIEIAIKLVKTELQSPEHPLDQHYRNLHCALRPLD
HESYEFKVISQYLQSTHAPTHSDYTMTLLDLFEVEKDGEKEAFREDLHNRMLLWHGSRMS
NWVGILSHGLRIAPPEAPITGYMFGKGIYFADMSSKSANYCFASRLKNTGLLLLSEVALG
QCNELLEANPKAEGLLQGKHSTKGLGKMAPSSAHFVTLNGSTVPLGPASDTGILNPDGYT
LNYNEYIVYNPNQVRMRYLLKVQFNFLQLW"""

ensembl_sequence= ''.join(ensembl_sequence.split())

In [150]:
uniprot_sequence = "MAARRRRSTGGGRARALNESKRVNNGNTAPEDSSPAKKTRRCQRQESKKMPVAGGKANKDRTEDKQDESVKALLLKGKAPVDPECTAKVGKAHVYCEGNDVYDVMLNQTNLQFNNNKYYLIQLLEDDAQRNFSVWMRWGRVGKMGQHSLVACSGNLNKAKEIFQKKFLDKTKNNWEDREKFEKVPGKYDMLQMDYATNTQDEEETKKEESLKSPLKPESQLDLRVQELIKLICNVQAMEEMMMEMKYNTKKAPLGKLTVAQIKAGYQSLKKIEDCIRAGQHGRALMEACNEFYTRIPHDFGLRTPPLIRTQKELSEKIQLLEALGDIEIAIKLVKTELQSPEHPLDQHYRNLHCALRPLDHESYEFKVISQYLQSTHAPTHSDYTMTLLDLFEVEKDGEKEAFREDLHNRMLLWHGSRMSNWVGILSHGLRIAPPEAPITGYMFGKGIYFADMSSKSANYCFASRLKNTGLLLLSEVALGQCNELLEANPKAEGLLQGKHSTKGLGKMAPSSAHFVTLNGSTVPLGPASDTGILNPDGYTLNYNEYIVYNPNQVRMRYLLKVQFNFLQLW"

In [151]:
ensembl_sequence == uniprot_sequence

True

In [184]:
transcript_mapping= pd.read_csv("data/transcriptID_mapping.csv")

In [175]:
#drop columns unnamed 5 and unnamed 6
transcript_mapping = transcript_mapping.drop(columns=['Unnamed: 5', 'Unnamed: 6'])


In [180]:
transcript_mapping.to_csv("data/transcriptID_mapping.csv", sep=',', index=False)

In [185]:
transcript_mapping

,GeneID,TranscriptID_Ensembl_isoform,UniProtID_isoform,is_canonical_in_uniprot,Protein_Sequence
0,EGFR,ENST00000275493.7,P00533-1,True,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
1,KRAS,ENST00000311936.8,P01116-2,False,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...
2,NRAS,ENST00000369535.5,P01111,True,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...
3,HRAS,ENST00000311189.8,P01112-1,True,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...
4,BRAF,ENST00000646891.2,P15056,True,MAALSGGGGGGAEPGQALFNGDMEPEAGAGAGAAASSAADPAIPEE...
5,RAF1,ENST00000251849.9,P04049-1,True,MEHIQGAWKTISNGFGFKDAVFDGSSCISPTIVQQFGYQRRASDDG...
6,ERBB3,ENST00000267101.8,P21860-1,True,MRANDALQVLGLLFSLARGSEVGNSQAVCPGTLNGLSVTGDAENQY...
7,ERBB2,ENST00000269571.10,P04626-1,True,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...
8,MAPK1,ENST00000215832.11,P28482-1,True,MAAAAAAGAGPEMVRGQVFDVGPRYTNLSYIGEGAYGMVCSAYDNV...
9,MAPK3,ENST00000263025.9,P27361-1,True,MAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQ...


In [ ]:
MC_EG_proteins_uniprot={
    'P00533',
    'P01116',
    'P01111',
    'P01112',
    'P15056',
    'P04049',
    'P21860',
    'P04626',
    'P28482',
    'P27361',
    'Q02750',
    'P36507',
    'P42336',
    'P42338',
    'O00329',
    'P31749',
    'P01106',
    'P29353',
    'P62993',
    'Q13480',
    'P26006',
    'P08069',
    'P21802',
    'P42345',
    'Q6R327',
    'P21359',
    'P60484',
    'P14618',
    'Q53GA4',
    'P52209',
    'P10415',
    'P09874',
    'Q9UGN5'
}

# Predicted Changes in Stability

In [ ]:
#run on server
foldx = pd.read_csv('/data/benchmarks/genomic_annot/afdb_foldx_export_20250210.csv') 

In [ ]:
foldx.shape

In [ ]:
foldx.head()

In [ ]:
#Check if uniprot_acession_proteins are in foldx dataset - 'True' if they are
set(MC_EG_proteins_uniprot).issubset(set(foldx['uniprot_accession'].unique()))

## Merge foldx_ddg with the annotated dataset

In [ ]:
#Load the transcript mapping ID dataset
transcriptID_mapping = pd.read_csv('/data/benchmarks/genomic_annot/transcriptID_mapping.csv', sep=';')

In [ ]:
transcriptID_mapping

In [ ]:
#Map Ensembl Transcript to uniprot accession
mapping_dict = dict(zip(transcriptID_mapping["TranscriptID_Ensembl_isoform"], transcriptID_mapping["UniProtID_global"]))
annotated_control_df["uniprot_accession"] = annotated_control_df["Target Transcript ID"].map(mapping_dict)

In [ ]:
# Change column position
cols = list(annotated_control_df.columns)

idx = cols.index("Target Transcript ID")
cols.remove("uniprot_accession")
cols.insert(idx + 1, "uniprot_accession")

annotated_control_df = annotated_control_df[cols]

In [ ]:
annotated_control_df.head(20)

#### Get wild type, mutated type and AA position from 'Amino Acid Edits'

In [ ]:
#Apply convert_amino_acid_variant to generate "H1109Y"-like string

# Create a mask for non-NC entries
mask = annotated_control_df["Amino Acid Edits"] != "(NC)"

# Apply conversion function only to valid entries
annotated_control_df.loc[mask, "aa_change"] = annotated_control_df.loc[mask, "Amino Acid Edits"].map(
    BECancerResistome.convert_amino_acid_variants
)

# Fill the others with "(NC)" if needed
annotated_control_df.loc[~mask, "aa_change"] = "(NC)"

In [ ]:
# Change column position
cols = list(annotated_control_df.columns)

idx = cols.index("Amino Acid Edits")
cols.remove("aa_change")
cols.insert(idx + 1, "aa_change")

annotated_control_df = annotated_control_df[cols]

In [ ]:
#Split aa_change into wild_type, position, mutated_type (with NC masking)

annotated_control_df.loc[mask, "wild_type"] = annotated_control_df.loc[mask, "aa_change"].str[0]
annotated_control_df.loc[mask, "uniprot_position"] = annotated_control_df.loc[mask, "aa_change"].str[1:-1].astype(int)
annotated_control_df.loc[mask, "mutated_type"] = annotated_control_df.loc[mask, "aa_change"].str[-1]

# Fill rest with NaN
annotated_control_df.loc[~mask, ["wild_type", "uniprot_position", "mutated_type"]] = np.nan

#### Merge foldx_ddg 

In [ ]:
annotated_foldx_control_df = annotated_control_df.merge(
    foldx[["uniprot_accession", "uniprot_position", "wild_type", "mutated_type", "foldx_ddg", "plddt"]],
    how="left",
    on=["uniprot_accession", "uniprot_position", "wild_type", "mutated_type"]
)

In [ ]:
annotated_foldx_control_df.head(20)

In [ ]:
#Filter rows with AA changes for missingness analysis 
annotated_foldx_control_df_aa = annotated_foldx_control_df[annotated_foldx_control_df["aa_change"] != "(NC)"]

In [ ]:
missing_values = annotated_foldx_control_df['foldx_ddg'].isna().sum()
percent_missing = 100 * missing_values / len(annotated_foldx_control_df)

print(f'Missing foldx ddg values in the dataset: {missing_values}')
print(f'Missing foldx ddg values in the dataset (%): {percent_missing}')

In [ ]:
#Missing values that are not because of the absence of aa change
missing_values_aa = annotated_foldx_control_df_aa['foldx_ddg'].isna().sum()
percent_missing_aa = 100 * missing_values_aa / len(annotated_foldx_control_df_aa)

print(f'Missing foldx ddg values in the dataset: {missing_values_aa}')
print(f'Missing foldx ddg values in the dataset (%): {percent_missing_aa}')

In [ ]:
annotated_foldx_control_df.to_csv("/data/benchmarks/genomic_annot/zscores-unambiguous-VEPannotated-processed-gene-expression-foldx-control")

In [ ]:
# Filter rows where foldx_ddg is missing
missing_foldx = annotated_foldx_control_df_aa[annotated_foldx_control_df_aa["foldx_ddg"].isna()]

# Get distribution of 'Hit_class' in those rows
hit_class_distribution = missing_foldx["Hit_class"].value_counts(dropna=False)

# Show both counts and percentages
hit_class_percent = missing_foldx["Hit_class"].value_counts(normalize=True, dropna=False) * 100

# Display nicely
distribution_df = pd.DataFrame({
    "Count": hit_class_distribution,
    "Percentage": hit_class_percent.round(2)
})

display(distribution_df)

In [ ]:
len(missing_foldx)

In [ ]:
# Count missing foldx_ddg values grouped by uniprot_accession
missing_by_uniprot = (
    annotated_foldx_control_df_aa[annotated_foldx_control_df_aa["foldx_ddg"].isna()]
    .groupby("uniprot_accession")
    .size()
    .reset_index(name="Missing_count")
    .sort_values(by="Missing_count", ascending=False)
)

display(missing_by_uniprot)

# Random tests

In [5]:
len(annotated_control_df)

147319

In [10]:
protvar_test_low_plddt = pd.read_csv("data/5d6e050dde2a1e688fb5a6403926e424-fun-GRCh38.csv")

In [11]:
protvar_test_low_plddt

,User_input,Chromosome,Coordinate,ID,Reference_allele,Alternative_allele,Notes,Gene,Codon_change,Strand,CADD_phred_like_score,Canonical_isoform_transcripts,MANE_transcript,Uniprot_canonical_isoform_(non_canonical),Alternative_isoform_mappings,Protein_name,Amino_acid_position,Amino_acid_change,Consequences,Residue_function_(evidence),Region_function_(evidence),Protein_existence_evidence,Protein_length,Entry_last_updated,Sequence_last_updated,Protein_catalytic_activity,Protein_complex,Protein_sub_cellular_location,Protein_family,Protein_interactions_PROTEIN(gene),Predicted_pockets(energy;per_vol;score;resids),Predicted_interactions(chainA-chainB;a_resids;b_resids;pDockQ),Foldx_prediction(foldxDdg;plddt),Conservation_score,AlphaMissense_pathogenicity(class),EVE_score(class),ESM1b_score,Genomic_location,Cytogenetic_band,Other_identifiers_for_the_variant,Diseases_associated_with_variant,Variants_colocated_at_residue_position,Position_in_structures
0,P01106 D2N,8,127736597,NaN,G,A,NaN,MYC,Gau/Aau,+,22.1,[ENSP00000478887.2(ENST00000621592.8)],NaN,P01106,P01106-3;2;Asp/Asn;missense;[ENSP00000430235.2...,Myc proto-oncogene protein,2,Asp/Asn,missense,CONFLICT-in Ref. 9; BAA01374(),CHAIN-Myc proto-oncogene protein;Range:1-454,Evidence at protein level,454,2025-02-05,2023-02-22,NaN,Efficient DNA binding requires dimerization wi...,"Nucleus, nucleoplasm;Nucleus, nucleolus;Nucleu...",NaN,Q9UBB4(ATXN10);O15169(AXIN1);O00499-10(BIN1);O...,NaN,NaN,foldxDdg:-0.23776;plddt:55.7,0.938,0.4683(AMBIGUOUS),NaN,-3.721,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
#non canonical case
annotated_control_df[annotated_control_df['Target Transcript ID'] == 'ENST00000311936.8'].head(10)

,Guide,Editor,Gene,Cell_Line,Drug,zscore,Hit_class,Source,Target Transcript ID,RefSeq match transcript (MANE Select),Amino Acid Edits,Mutation_Category_enc,IMPACT_enc,TSL,SIFT_pathogenicity,PolyPhen_pathogenicity,REVEL,ClinPred,EVE_SCORE,AlphaMissense_score,BayesDel_noAF_score,DANN_score,DEOGEN2_score,ESM1b_score,Eigen-PC-phred_coding,Eigen-PC-raw_coding,GERP++_NR,GERP++_RS,LIST-S2_score,MPC_rankscore,MPC_score,MVP_score,MetaRNN_score,MetaSVM_score,MutFormer_score,MutationAssessor_score,PROVEAN_score,PrimateAI_pred_enc,VARITY_R_LOO_score,bStatistic,fathmm-XF_coding_score,gMVP_score,phastCons100way_vertebrate,phyloP100way_vertebrate,CADD_PHRED,MaxEntScan_alt,MaxEntScan_diff,MaxEntScan_ref,SpliceAI_pred_DP_AG,SpliceAI_pred_DP_AL,SpliceAI_pred_DP_DG,SpliceAI_pred_DP_DL,SpliceAI_pred_DS_AG,SpliceAI_pred_DS_AL,SpliceAI_pred_DS_DG,SpliceAI_pred_DS_DL,BLOSUM62,LOEUF,mutfunc_exp,ada_score,Gene_expression_voom
77,AAAATGACTGAATATAAACT,CBE,KRAS,A375,PIC,-0.369801,non-hit,NaN,ENST00000311936.8,NM_004985.5,Thr2Ile,3.0,2.0,1.0,0.00,NaN,0.648,0.917700,NaN,0.7771,0.279860,0.996342,NaN,-11.051,6.596012,0.628262,5.68,5.68,NaN,0.87961,NaN,0.981930,0.723242,0.7167,0.997969,0.715,-3.84,0.0,NaN,707.0,0.914873,NaN,1.0,9.985,27.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,0.264,-0.08422,NaN,5.69374
132,AAACTTGTGGTAGTTGGAGC,CBE,KRAS,A375,PIC,-0.142688,non-hit,NaN,ENST00000311936.8,NM_004985.5,Leu6Phe,3.0,2.0,1.0,0.00,NaN,0.657,0.922023,0.593773,0.9718,0.222443,0.999069,NaN,-11.807,8.686395,0.745069,5.68,5.68,NaN,0.98879,NaN,0.983524,0.778710,0.3591,0.998665,1.740,-2.89,0.0,NaN,707.0,0.762611,NaN,1.0,6.867,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.264,4.38564,NaN,5.69374
138,AAAGACAAAGTGTGTAATTA,CBE,KRAS,A375,PIC,0.924225,non-hit,NaN,ENST00000311936.8,NM_004985.5,Thr183Ile,3.0,2.0,1.0,0.45,NaN,0.475,0.422254,NaN,0.1642,0.316470,0.953102,NaN,-5.401,7.975373,0.711608,5.79,5.79,NaN,NaN,NaN,0.992356,0.610068,0.7353,0.973412,NaN,0.33,NaN,NaN,874.0,0.847446,NaN,1.0,9.546,23.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,0.264,NaN,NaN,5.69374
277,AACAGGCTCAGGACTTAGCA,CBE,KRAS,A375,PIC,-0.558241,non-hit,NaN,ENST00000311936.8,NM_004985.5,Ala130Val,3.0,2.0,1.0,0.04,NaN,0.633,0.982892,0.214970,0.6821,0.187503,0.998313,NaN,-8.078,6.684818,0.634569,5.52,5.52,NaN,0.09867,NaN,0.950875,0.843032,-0.1246,0.995482,1.410,-2.45,0.0,NaN,861.0,0.902354,NaN,1.0,9.984,29.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.264,3.51754,NaN,5.69374
289,AACATCAGCAAAGACAAGAC,CBE,KRAS,A375,PIC,0.053361,non-hit,NaN,ENST00000311936.8,NM_004985.5,Ser145Leu,3.0,2.0,1.0,0.00,NaN,0.860,NaN,0.771730,0.9984,0.480580,0.998796,NaN,-16.992,26.179710,1.091980,5.52,5.52,NaN,0.39068,NaN,0.995916,0.928741,1.0856,0.999324,4.685,-5.47,0.0,NaN,861.0,0.916834,NaN,1.0,9.984,33.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.0,0.264,4.35318,NaN,5.69374
431,AAGAAGTCAAAGACAAAGTG,CBE,KRAS,A375,PIC,0.210527,non-hit,NaN,ENST00000311936.8,NM_004985.5,Ser181Leu,3.0,2.0,1.0,0.44,NaN,0.339,NaN,NaN,0.1080,0.088685,0.991925,NaN,-6.395,4.608434,0.433976,5.79,5.79,NaN,NaN,NaN,0.977802,0.269154,0.3332,0.916115,NaN,-0.11,NaN,NaN,874.0,0.866388,NaN,1.0,9.546,24.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.0,0.264,NaN,NaN,5.69374
485,AAGCAAGTAGTAATTGATGG,CBE,KRAS,A375,PIC,1.081328,non-hit,NaN,ENST00000311936.8,NM_004985.5,Gln43Ter,4.0,3.0,1.0,NaN,NaN,NaN,0.968550,NaN,NaN,0.655873,0.998189,NaN,NaN,30.959970,1.130018,5.51,5.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,855.0,0.241387,NaN,1.0,9.985,51.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.264,1.56822,NaN,5.69374
674,AATAATACTAAATCATTTGA,CBE,KRAS,A375,PIC,0.725648,non-hit,NaN,ENST00000311936.8,NM_004985.5,Thr87Ile,3.0,2.0,1.0,0.07,NaN,0.217,0.854378,0.105601,0.1499,-0.217405,0.931753,NaN,-6.488,2.598354,0.065765,5.76,4.87,NaN,0.87690,NaN,0.884393,0.310494,-0.8189,0.975132,-0.050,-2.09,0.0,NaN,856.0,0.733983,NaN,1.0,6.883,24.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,0.264,-0.03424,NaN,5.69374
745,AATGACTGAATATAAACTTG,CBE,KR

In [16]:
protvar_test_alternative_isoforms = pd.read_csv("data/e27e9a6751f7a183b53d18587369e5bf-fun-pop-str-GRCh38.csv")

In [17]:
protvar_test_alternative_isoforms

,User_input,Chromosome,Coordinate,ID,Reference_allele,Alternative_allele,Notes,Gene,Codon_change,Strand,CADD_phred_like_score,Canonical_isoform_transcripts,MANE_transcript,Uniprot_canonical_isoform_(non_canonical),Alternative_isoform_mappings,Protein_name,Amino_acid_position,Amino_acid_change,Consequences,Residue_function_(evidence),Region_function_(evidence),Protein_existence_evidence,Protein_length,Entry_last_updated,Sequence_last_updated,Protein_catalytic_activity,Protein_complex,Protein_sub_cellular_location,Protein_family,Protein_interactions_PROTEIN(gene),Predicted_pockets(energy;per_vol;score;resids),Predicted_interactions(chainA-chainB;a_resids;b_resids;pDockQ),Foldx_prediction(foldxDdg;plddt),Conservation_score,AlphaMissense_pathogenicity(class),EVE_score(class),ESM1b_score,Genomic_location,Cytogenetic_band,Other_identifiers_for_the_variant,Diseases_associated_with_variant,Variants_colocated_at_residue_position,Position_in_structures
0,P01116 S181L,12,25215470,NaN,C,A,WARN:User input reference amino acid (Ser) doe...,KRAS,Gug/Uug,-,22.6,[ENSP00000256078.5(ENST00000256078.10)],NaN,P01116,L7RSL8;181;Val/Leu;missense;[ENSP00000256078.5...,GTPase KRas,181,Val/Leu,missense,NaN,CHAIN-GTPase KRas;Range:1-186|CHAIN-GTPase KRa...,Evidence at protein level,189,2025-02-05,1986-07-21,RHEA:19669(PubMed:[20949621]),Interacts with PHLPP (By similarity)|Interacts...,"Cell membrane;Endomembrane system;Cytoplasm, c...",Belongs to the small GTPase superfamily,Q96II5(ARAF);Q99755(PIP5K1A);P04049(RAF1);P507...,NaN,NaN,foldxDdg:-0.272648;plddt:51.94,0.336,0.0979(BENIGN),NaN,-1.963,NaN,NaN,NaN,NaN,[Val>Ala;NC_000012.12:g.25215469A>G;12p12.1;En...,6pts;C181;0.0;Solution NMR|6ptw;C181;0.0;Solut...
1,P01116 S181L,12,25215470,NaN,C,G,WARN:User input reference amino acid (Ser) doe...,KRAS,Gug/Cug,-,22.6,[ENSP00000256078.5(ENST00000256078.10)],NaN,P01116,L7RSL8;181;Val/Leu;missense;[ENSP00000256078.5...,GTPase KRas,181,Val/Leu,missense,NaN,CHAIN-GTPase KRas;Range:1-186|CHAIN-GTPase KRa...,Evidence at protein level,189,2025-02-05,1986-07-21,RHEA:19669(PubMed:[20949621]),Interacts with PHLPP (By similarity)|Interacts...,"Cell membrane;Endomembrane system;Cytoplasm, c...",Belongs to the small GTPase superfamily,Q96II5(ARAF);Q99755(PIP5K1A);P04049(RAF1);P507...,NaN,NaN,foldxDdg:-0.272648;plddt:51.94,0.336,0.0979(BENIGN),NaN,-1.963,NaN,NaN,NaN,NaN,[Val>Ala;NC_000012.12:g.25215469A>G;12p12.1;En...,6pts;C181;0.0;Solution NMR|6ptw;C181;0.0;Solut...


In [19]:
print(protvar_test_alternative_isoforms.loc[1, "Alternative_isoform_mappings"])

L7RSL8;181;Val/Leu;missense;[ENSP00000256078.5(ENST00000256078.10)]|P01116-1;181;Val/Leu;missense;[ENSP00000256078.5(ENST00000256078.10)]
